# A3T-GCN ablation (E0-E8) on Kaggle

E0 = A3T-GCN (road graph), E1 = + VAR, E3 = + VAR + PEM (the former E2), NOGRAPH = no-graph
baseline; E2 and E4-E7 are earlier phases (`docs/architecture.md`).

**E8 / E8V (VAR-PEM impulse-response graphs, no LiNGAM)**
- `E8V`: S_h = |Psi_h| (VAR only, no PEM), top-25 sources per target.
- `E8`: S_h = P * |Psi_h|, where P comes from the PEM run with a traffic-correlation prior.
  Prior-study variants (`GRAPH_VARIANT`): `flow_nu0_0.01_nu1_0.2` (proposed), `uniform_nu0_0.01_nu1_0.2` (no prior),
  `flow_shuffle_nu0_0.01_nu1_0.2` (control: same prior values, shuffled pairs).
- PEM counts as useful only if E8 beats E8V; the correlation prior counts only if flow beats flow_shuffle and uniform.
  Judge differences against the spread over seeds (42, 7, 123), not against one run.

Only the graph differs between runs; all hyperparameters are fixed in `config.py` of the cloned repository and verified in section 2b.
Training needs only the finished graph files, never the PEM / VAR / LiNGAM result files.

**Before running**
1. *Add Input* → the GBA dataset (`gba_his_raw_2021.h5`, `gba_rn_adj.npy`, `gba_meta.csv`).
2. *Add Input* → the artifacts dataset: the graph `.npz` files you need (`E1__*`, `E3__*`, ..., `E8__h{1,3,6,12}__*`, `E8V__h{1,3,6,12}__*`)
   and both `sensor_filter_*.json` files. Kaggle drops "+" from file names; section 2 restores it.
3. Settings: **Accelerator = GPU P100**, **Internet = On**, and keep the **same environment** for every experiment.
4. Change only `EXPERIMENT`, `GRAPH_VARIANT`, `HORIZON` and `SEED` in the settings cell, then **Save Version → Save & Run All (Commit)**.
   Only commit runs keep their files: download `output_<RUN_NAME>` from that version's *Output* tab.
5. **Repeat test (optional, to check that training is repeatable):** set `EPOCHS = 2` and `REPEAT_TEST = True`. The same command then runs twice in one session (about 40 minutes each) and section 3b prints both runs epoch by epoch.


## Run settings (change only EXPERIMENT, GRAPH_VARIANT, HORIZON, SEED)

In [ ]:
# Final runs: change ONLY the four lines marked CHANGE. Everything else follows from them.
EXPERIMENT     = "E8"      # CHANGE: "E0" ... "E8", "E8V" or "NOGRAPH" (docs/architecture.md); a string, with quotes

# Graph file per experiment (None = built in). The old E2__ file of the
# VAR + PEM graph is still accepted for E3 (same graph).
FINAL_GRAPHS = {   # same names as config.FINAL_GRAPHS (build_graph.py output)
    "E0": None,
    "NOGRAPH": None,
    "E1": "E1__lag3_alpha1e+06.npz",
    "E2": "E2__lag3_alpha1e+06__rows10000.npz",
    # E3 amended 2026-10-28: E1 VAR weights kept on PEM edges (the old P*D graph is refused)
    "E3": "E3__lag3_alpha1e+06__lag3_alpha1e+06__nu0_0.0017_nu1_1_tau_0.01_thr_0.5_eta_0.5_L-BFGS-B_std.npz",
    "E4": "E4__lag3_alpha1e+06__rows10000__lag3_alpha1e+06__nu0_0.0017_nu1_1_tau_0.01_thr_0.5_eta_0.5_L-BFGS-B_std.npz",
    "E5": "E5__h{h}__lag3_alpha1e+06__rows10000__lag3_alpha1e+06__nu0_0.0017_nu1_1_tau_0.01_thr_0.5_eta_0.5_L-BFGS-B_std.npz",
    "E6": "E6__lag3_alpha1e+06__cand-corr_k25__rows10000__lag3_alpha1e+06__cand-corr_k25__nu0_0.0017_nu1_1_tau_0.01_thr_0.5_eta_0.5_L-BFGS-B_std.npz",
    "E7": "E7__lag3_alpha1e+06__rows10000__lag3_alpha1e+06__nu0_0.0017_nu1_1_tau_0.01_thr_0.5_eta_road0.7-0.3_L-BFGS-B_std.npz",
    # E8 (VAR-PEM, no LiNGAM, S_h = P * |Psi_h|) and E8V (ablation, S_h = |Psi_h|): one file per horizon (1, 3, 6, 12).
    # The "E8" entry below is the old default PEM (nu0 0.0017, nu1 1); the E8 prior study uses GRAPH_VARIANT instead.
    "E8": "E8__h{h}__lag3_alpha1e+06__lag3_alpha1e+06__nu0_0.0017_nu1_1_tau_0.01_thr_0.5_eta_0.5_L-BFGS-B_std.npz",
    "E8V": "E8V__h{h}__lag3_alpha1e+06.npz",
}

# Forecast horizon in 5-minute steps: 1, 3, 6 or 12 (5 / 15 / 30 / 60 min).
# E5, E8 and E8V need one graph file per horizon ("{h}" in their names); E1-E4, E6, E7 keep one graph.
HORIZON        = 1         # CHANGE: 1, 3, 6 or 12

# E8 prior study (built by e8-prior-study.ipynb): which E8 graph to use. Must be a string or None.
#   "flow_nu0_0.01_nu1_0.2"        proposed: prior from the detrended traffic-flow correlation
#   "uniform_nu0_0.01_nu1_0.2"     no prior (eta = 0.5 everywhere)
#   "flow_shuffle_nu0_0.01_nu1_0.2" control: the flow prior values shuffled over the road pairs
#   None                           standard E8 file above (not part of the prior study)
# Only valid with EXPERIMENT = "E8" (E8V does not depend on PEM: keep None for it and for every other experiment).
GRAPH_VARIANT  = "flow_nu0_0.01_nu1_0.2"   # CHANGE
KNOWN_E8_VARIANTS = {None, "flow_nu0_0.01_nu1_0.2", "uniform_nu0_0.01_nu1_0.2", "flow_shuffle_nu0_0.01_nu1_0.2"}
assert GRAPH_VARIANT in KNOWN_E8_VARIANTS, f"unknown GRAPH_VARIANT {GRAPH_VARIANT!r}: use one of {sorted(map(str, KNOWN_E8_VARIANTS))}"
assert GRAPH_VARIANT is None or EXPERIMENT == "E8", "GRAPH_VARIANT only applies to EXPERIMENT = 'E8' (set it to None otherwise)"
assert HORIZON in (1, 3, 6, 12), "HORIZON must be 1, 3, 6 or 12"
if GRAPH_VARIANT:
    FINAL_GRAPHS["E8"] = f"E8__h{{h}}__lag3_alpha1e+06__{GRAPH_VARIANT}.npz"

GRAPH_FILE     = FINAL_GRAPHS[EXPERIMENT]
GRAPH_FILE     = None if GRAPH_FILE is None else GRAPH_FILE.format(h=HORIZON)

# Random seed. Main runs use 42; the repeat seeds of the E8 study are 7 and 123 (only the random initial weights
# change). Report every run: a difference between graphs is only meaningful against this seed-to-seed spread.
SEED           = 42        # CHANGE: 42, 7 or 123

# Shared settings are frozen by Phase 3 (lr 0.001, 20 epochs): leave None.
EPOCHS         = None      # None = 20 epochs from config.py
LEARNING_RATE  = None      # None = 0.001 from config.py
MILESTONES     = None      # e.g. [20]: also keep the best checkpoint of epochs <= 20
SELECTION_ONLY = False     # False = the test set is evaluated once at the end
MAX_HOURS      = 8.5       # keep below the GPU hours you have left this week (the run stops itself at this limit)
# Diagnostic (use with EPOCHS = 2): run the same command a second time in this session and compare the two
# runs epoch by epoch (section 3b). Identical numbers = training is repeatable here; different = it is not.
REPEAT_TEST    = False

# Phase 3 (done, 2026-10-27): kept lr 0.001 / 20 epochs. The reference is the finished E0 run
# (output_E0_final: lr 0.001, 20 epochs). Final runs always use EPOCHS = LEARNING_RATE = MILESTONES = None,
# SELECTION_ONLY = False (or the settings Phase 3 selects, once they are written into config.py).

RUN_NAME = (
    EXPERIMENT
    + ("" if not GRAPH_VARIANT else f"_{GRAPH_VARIANT}")
    + ("" if LEARNING_RATE is None else f"_lr{LEARNING_RATE:g}")
    + ("" if EPOCHS is None else f"_{EPOCHS}ep")
    + ("" if HORIZON == 1 else f"_h{HORIZON}")
    + ("_sel" if SELECTION_ONLY else "_final")
    + ("" if SEED == 42 else f"_seed{SEED}")
)

REPO_URL       = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH    = "main"

# Data and artifacts are found automatically under /kaggle/input (section 2).
print("Experiment:", EXPERIMENT, "| variant:", GRAPH_VARIANT, "| horizon:", HORIZON, "| graph:", GRAPH_FILE,
      "| seed:", SEED, "| run name:", RUN_NAME)


## 1. Environment (Kaggle's built-in TensorFlow; note the versions - keep them identical for E0, E1, E2)

In [ ]:
import sys
print(sys.version)
print(sys.version_info)

In [ ]:
import sys
import tensorflow as tf
import numpy as np
import scipy
import h5py
import pandas as pd

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("h5py:", h5py.__version__)
print("Pandas:", pd.__version__)

print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Data, code (fresh clone) and artifacts

In [ ]:
import os
import glob
import shutil
import subprocess

# Folder under /kaggle/input that contains the LargeST-GBA files
# (e.g. /kaggle/input/datasets/sanjumurali/gba-dataset).
hits = glob.glob("/kaggle/input/**/gba_his_raw_2021.h5", recursive=True)
assert hits, "gba_his_raw_2021.h5 not found under /kaggle/input - add the GBA dataset (Add Input)"
DATA_DIR = os.path.dirname(hits[0])

print("DATA_DIR:", DATA_DIR)

required_files = [
    "gba_his_raw_2021.h5",
    "gba_rn_adj.npy",
    "gba_meta.csv",
]

for name in required_files:
    path = os.path.join(DATA_DIR, name)
    assert os.path.exists(path), f"Missing dataset file: {path}"
    print("✓", path)


In [ ]:
WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"  # package name must be 'implementation'

# ------------------------------------------------------------------
# Clone the implementation repository
# ------------------------------------------------------------------
if os.path.exists(CODE):
    shutil.rmtree(CODE)

subprocess.run(
    ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE],
    check=True
)

print(
    subprocess.run(
        ["git", "-C", CODE, "log", "-1", "--oneline"],
        capture_output=True,
        text=True
    ).stdout
)

# ------------------------------------------------------------------
# Dataset: link DATA_DIR (previous cell) to the location expected by
# the implementation: implementation/datasets/LargeST GBA/
# ------------------------------------------------------------------
DATA_TARGET = f"{CODE}/datasets/LargeST GBA"

os.makedirs(f"{CODE}/datasets", exist_ok=True)

if os.path.lexists(DATA_TARGET):
    if os.path.islink(DATA_TARGET):
        os.unlink(DATA_TARGET)
    else:
        shutil.rmtree(DATA_TARGET)

os.symlink(DATA_DIR, DATA_TARGET)

for name in required_files:
    path = os.path.join(DATA_TARGET, name)
    assert os.path.exists(path), f"Missing dataset file: {path}"

print("Dataset linked successfully:")
for name in required_files:
    print("  ✓", name)

# ------------------------------------------------------------------
# Artifacts: graph files and the sensor-filter cache, found
# anywhere under /kaggle/input (no fixed dataset path needed)
# ------------------------------------------------------------------
os.makedirs(f"{CODE}/outputs/graphs", exist_ok=True)
os.makedirs(f"{CODE}/outputs/sensor_filter", exist_ok=True)

for path in glob.glob("/kaggle/input/**/*", recursive=True):
    if not os.path.isfile(path):
        continue
    filename = os.path.basename(path)
    if filename.startswith(tuple(f"E{i}__" for i in range(1, 9)) + ("E8V__",)) and filename.endswith((".npz", ".json")):
        # Kaggle drops "+" from uploaded file names (alpha1e+06 -> alpha1e06);
        # restore the original name expected by config.py.
        original = filename.replace("alpha1e06", "alpha1e+06")
        shutil.copy(path, f"{CODE}/outputs/graphs/{original}")
    elif filename.startswith("sensor_filter_") and filename.endswith(".json"):
        shutil.copy(path, f"{CODE}/outputs/sensor_filter/")

print("graphs:", sorted(os.listdir(f"{CODE}/outputs/graphs")))
print("sensor filter cache:", os.listdir(f"{CODE}/outputs/sensor_filter"))

assert GRAPH_FILE is None or os.path.exists(f"{CODE}/outputs/graphs/{GRAPH_FILE}"), \
    f"{GRAPH_FILE} not found - add the artifacts dataset (Add Input)"


## 2b. Fixed hyperparameters (identical for every experiment): verified, do not edit

In [ ]:
# Fixed hyperparameters of the ablation: identical for every experiment.
# The run stops here if the cloned code does not contain exactly these values.
EXPECTED = {
    # shared A3T-GCN / training settings
    "learning_rate": 0.001,
    "grad_clip_norm": 5.0,
    "training_epoch": 20,
    "batch_size": 8,
    "gru_units": 100,
    "seq_len": 12,
    "pre_len": 1,
    "lambda_loss": 0.0015,
    "lr_patience": 5,
    "lr_factor": 0.5,
    "train_rate": 0.8,
    "val_rate": 0.1,
    "seed": 42,              # code default, always 42; this run's seed is SEED (passed as --seed)
    "sensor_filter": "duplicates",
    "imputation": "gap_bands",
    # component settings (graph experiments). PEM_NU0 / PEM_NU1 are the old defaults in config.py; the E8 prior-study
    # graphs (GRAPH_VARIANT) were built with nu0 0.01 / nu1 0.2, which is recorded in the graph file name, not here.
    "VAR_LAG": 3,
    "VAR_ALPHA": 1e6,
    "PEM_NU0": 0.0017,
    "PEM_NU1": 1.0,
    "PEM_STANDARDISE": True,
    "LINGAM_MAX_ROWS": 10000,
}

import importlib
import sys
sys.path.insert(0, WORK)
cfg_module = importlib.import_module("implementation.config.config")
# Use the graph file chosen in the settings cell (GRAPH_VARIANT runs have their own file name).
cfg = cfg_module.ExperimentConfig(experiment=EXPERIMENT, horizon=HORIZON,
                                  graph_path=None if GRAPH_FILE is None else f"{CODE}/outputs/graphs/{GRAPH_FILE}")

actual = {k: getattr(cfg, k) if hasattr(cfg, k) else getattr(cfg_module, k) for k in EXPECTED}
mismatch = {k: (actual[k], EXPECTED[k]) for k in EXPECTED if actual[k] != EXPECTED[k]}

for k, v in actual.items():
    print(f"{k:16s} {v}")
if EPOCHS is not None or LEARNING_RATE is not None:
    print(f"NOTE: override EPOCHS={EPOCHS}, LEARNING_RATE={LEARNING_RATE} "
          "(Phase 3 selection runs only; final runs use the frozen settings)")

assert not mismatch, f"Cloned code has different hyperparameters (actual, expected): {mismatch}. Push the latest code."
cfg.validate()

# The graph that will be trained on must be exactly the one chosen in the settings cell.
if GRAPH_FILE is not None:
    assert os.path.basename(str(cfg.graph_path)) == GRAPH_FILE, (cfg.graph_path, GRAPH_FILE)
    if GRAPH_VARIANT:
        assert GRAPH_VARIANT in str(cfg.graph_path) and f"__h{HORIZON}__" in str(cfg.graph_path), cfg.graph_path
    if EXPERIMENT in ("E5", "E8", "E8V"):
        assert f"__h{HORIZON}__" in os.path.basename(str(cfg.graph_path)) or f"E5__h{HORIZON}__" in str(cfg.graph_path), \
            f"graph file is not the one for horizon {HORIZON}: {cfg.graph_path}"

print("\nFixed hyperparameters verified for", EXPERIMENT, "| variant:", GRAPH_VARIANT, "| horizon:", HORIZON)
print("Graph that will be used:", cfg.graph_path)
print("Seed used for this run:", SEED, "(main runs: 42; repeats: 7, 123)")


## 3. Run

In [ ]:

def build_cmd(run_name):
    cmd = [sys.executable, "-u", "-m", "implementation.main", "--experiment", EXPERIMENT]
    if GRAPH_FILE:
        cmd += ["--graph", f"{CODE}/outputs/graphs/{GRAPH_FILE}"]
    if EPOCHS is not None:
        cmd += ["--epochs", str(EPOCHS)]
    if LEARNING_RATE is not None:
        cmd += ["--learning-rate", str(LEARNING_RATE)]
    if MILESTONES:
        cmd += ["--milestones", *map(str, MILESTONES)]
    if HORIZON != 1:
        cmd += ["--horizon", str(HORIZON)]
    if MAX_HOURS:
        cmd += ["--max-hours", str(MAX_HOURS)]
    if SELECTION_ONLY:
        cmd += ["--selection-only"]
    if run_name:
        cmd += ["--run-name", run_name]
    cmd += ["--seed", str(SEED)]
    return cmd


def launch(cmd):
    print(" ".join(cmd))
    env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2", PYTHONUNBUFFERED="1")
    with subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            if not any(noise in line for noise in ("WARNING:tensorflow", "Instructions for updating", "non-resource variables")):
                print(line, end="")
    assert process.returncode == 0, f"main.py exited with {process.returncode}"


launch(build_cmd(RUN_NAME))


## 3b. Repeat test (only when REPEAT_TEST = True)

Runs the identical command a second time and compares the two runs epoch by epoch.
Use `EPOCHS = 2` (about 40 minutes per run). Nothing else in the notebook changes.

In [ ]:
import json
import pandas as pd

# Epoch-1 train loss of the two earlier E5 runs (horizon 1, seed 42, same settings): only meaningful for that run.
EARLIER_EPOCH1_LOSS = {"saved E5 run (test 28.17)": 168.90827182326598, "E5 rerun (test 35.53)": 140.761621}

if not REPEAT_TEST:
    print("REPEAT_TEST = False: skipped.")
else:
    first = RUN_NAME or EXPERIMENT
    second = f"{first}_repeat"
    launch(build_cmd(second))

    a = pd.read_csv(f"{CODE}/results/{first}/history.csv")
    b = pd.read_csv(f"{CODE}/results/{second}/history.csv")
    n = min(len(a), len(b))
    table = pd.DataFrame({
        "epoch": a["epoch"][:n],
        "train_loss_run1": a["train_loss"][:n], "train_loss_run2": b["train_loss"][:n],
        "val_rmse_run1": a["val_rmse_original"][:n], "val_rmse_run2": b["val_rmse_original"][:n],
    })
    table["loss_difference"] = (table["train_loss_run1"] - table["train_loss_run2"]).abs()
    pd.set_option("display.width", 200)
    print(table.to_string(index=False))

    identical = bool((table["loss_difference"] == 0).all()
                     and (table["val_rmse_run1"] == table["val_rmse_run2"]).all())
    if identical:
        verdict = "IDENTICAL: training is repeatable in this environment."
    elif float(table["loss_difference"].max()) < 1e-3 * float(table["train_loss_run1"].abs().max()):
        verdict = "NEARLY identical (rounding-level differences)."
    else:
        verdict = ("DIFFERENT: the same command gives different runs here, "
                   f"largest epoch-1 loss difference {float(table['loss_difference'].iloc[0]):.4f}.")
    print("\nVERDICT:", verdict)

    first_loss = float(a["train_loss"].iloc[0])
    for name, value in EARLIER_EPOCH1_LOSS.items():
        print(f"epoch-1 train loss of run 1 vs {name}: {first_loss:.6f} vs {value:.6f} "
              f"-> {'MATCH' if abs(first_loss - value) < 1e-3 else 'no match'}")

    json.dump({"verdict": verdict, "identical": identical, "table": table.to_dict(orient="list")},
              open(f"{WORK}/repeat_test.json", "w"), indent=2)
    shutil.copytree(f"{CODE}/results/{second}", f"{WORK}/output_{second}", dirs_exist_ok=True)
    print("Saved repeat_test.json and output_" + second)

## 4. Results (saved as notebook output)

In [ ]:
import json, os, shutil

# Uses EXPERIMENT / RUN_NAME from the settings cell.
run_dir = f"{CODE}/results/{RUN_NAME or EXPERIMENT}"
print("Looking in:", run_dir, "->", os.path.isdir(run_dir))
assert os.path.isdir(run_dir), "Run folder not found - did section 3 finish without errors?"

print("Files:", sorted(os.listdir(run_dir)))
results = json.load(open(f"{run_dir}/results.json"))
summary = {k: results.get(k) for k in ["experiment", "best_epoch", "best_val_rmse_original",
                                         "training_seconds", "valid_sequences", "test_metrics"]}
summary["learning_rate"] = results["config"].get("learning_rate")
summary["grad_clip_norm"] = results["config"].get("grad_clip_norm")
summary["graph"] = results.get("graph", {}).get("source")
summary["graph_variant"] = GRAPH_VARIANT
summary["horizon"] = HORIZON
summary["seed"] = SEED
summary["graph_file_used"] = os.path.basename(str(results.get("graph", {}).get("path", ""))) or None
print(json.dumps(summary, indent=2))
print("Checkpoint check:", results.get("checkpoint_check"))

# The run must have trained on the graph chosen in the settings cell (a wrong file would silently mislabel the result).
# Checked only when results.json records the graph path; the output is still saved below if it is not recorded.
if GRAPH_FILE is not None and summary["graph_file_used"] is not None:
    assert summary["graph_file_used"] == GRAPH_FILE, \
        f"run used {summary['graph_file_used']!r}, but the settings chose {GRAPH_FILE!r}"
elif GRAPH_FILE is not None:
    print("NOTE: results.json has no graph path; the graph was verified in section 2b:", GRAPH_FILE)

out = f"{WORK}/output_{RUN_NAME or EXPERIMENT}"
shutil.copytree(run_dir, out, dirs_exist_ok=True)
print("Saved to", out, "->", sorted(os.listdir(out)))

# Record what this run was made with (results.json does not hold it), so two runs can be compared later.
def _shell(command):
    try:
        return subprocess.run(command, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as error:            # nvidia-smi / git may be missing
        return f"unavailable: {error}"

try:
    build = {k: str(v) for k, v in tf.sysconfig.get_build_info().items()
             if k in ("cuda_version", "cudnn_version", "is_cuda_build")}
except Exception as error:
    build = {"unavailable": str(error)}

environment = {
    "python": sys.version, "tensorflow": tf.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
    "h5py": h5py.__version__, "pandas": pd.__version__, "tensorflow_build": build,
    "gpu": _shell(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"]),
    "visible_gpu_count": len(tf.config.list_physical_devices("GPU")),
    "cpu_count": os.cpu_count(),
    "code_commit": _shell(["git", "-C", CODE, "log", "-1", "--format=%H %ad %s"]),
    "run": {"experiment": EXPERIMENT, "graph_variant": GRAPH_VARIANT, "horizon": HORIZON, "seed": SEED,
            "epochs": EPOCHS, "run_name": RUN_NAME, "graph_file": GRAPH_FILE},
}
json.dump(environment, open(f"{out}/environment.json", "w"), indent=2)
print("environment:", json.dumps({k: environment[k] for k in ("tensorflow", "gpu", "code_commit")}, indent=1))

if not SELECTION_ONLY:
    assert os.path.exists(f"{out}/test_predictions.npz"), "test_predictions.npz missing!"
assert os.path.exists(f"{out}/epoch_metrics.csv"), "epoch_metrics.csv missing!"
assert os.path.exists(f"{out}/run_summary.json"), "run_summary.json missing!"
print("Download this folder from the version's Output tab:", out)


## 5. Visualization of this run

Training curve; for final runs also test metrics, MAE by time of day, per-sensor MAE and observed vs predicted. Figures are saved into the output folder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
try:
    from IPython.display import display
except ImportError:          # plain Python (outside a notebook)
    display = print

BLUE, INK, MUTED, GRID, SURFACE = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False,
})

label = RUN_NAME or EXPERIMENT
history = pd.read_csv(f"{out}/history.csv") if os.path.exists(f"{out}/history.csv") \
    else pd.DataFrame(results["history"])

# 1. Training curve: validation RMSE / MAE (top), learning rate (bottom, own axis)
fig, (ax, ax_lr) = plt.subplots(2, 1, figsize=(7.5, 5), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})
ax.plot(history["epoch"], history["val_rmse_original"], color=BLUE, marker="o", ms=4, label="Val RMSE")
ax.plot(history["epoch"], history["val_mae_original"], color=BLUE, ls="--", marker="o", ms=3, label="Val MAE")
best = history["val_rmse_original"].idxmin()
ax.plot(history["epoch"][best], history["val_rmse_original"][best], "o", ms=10,
        mfc=SURFACE, mec=BLUE, mew=2, label=f"best epoch {int(history['epoch'][best])}")
ax.set_ylabel("Vehicles / 5 min")
ax.set_title(f"{label}: validation error per epoch")
ax.legend()
ax_lr.step(history["epoch"], history["learning_rate"], where="post", color=MUTED)
ax_lr.set_ylabel("LR")
ax_lr.set_xlabel("Epoch")
ax_lr.xaxis.set_major_locator(MaxNLocator(integer=True))
fig.tight_layout(); fig.savefig(f"{out}/fig_training_curve.png", dpi=200); plt.show()

if not SELECTION_ONLY and os.path.exists(f"{out}/test_predictions.npz"):

    # 2. Test metrics table
    metrics = {k: v for k, v in results["test_metrics"].items() if k != "mape_min_flow"}
    table = pd.DataFrame([metrics], index=[label])
    table["best_epoch"] = results["best_epoch"]
    table["best_val_rmse"] = results["best_val_rmse_original"]
    table["training_h"] = results["training_seconds"] / 3600
    display(table.round(4))
    display(pd.DataFrame(results["test_breakdown"]).T.round(4))

    pred = np.load(f"{out}/test_predictions.npz")
    per_sensor = pd.read_csv(f"{out}/per_sensor_metrics.csv")
    tod = pd.read_csv(f"{out}/time_of_day_metrics.csv")

    # 3. MAE by time of day and per-sensor MAE distribution
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
    a1.plot(tod["slot"] * 5 / 60, tod["mae"].rolling(3, center=True, min_periods=1).mean(), color=BLUE)
    a1.set_xlim(0, 24); a1.set_xticks(range(0, 25, 3))
    a1.set_xlabel("Time of day (h)"); a1.set_ylabel("Test MAE (veh / 5 min)")
    a1.set_title("Test MAE by time of day")
    a2.hist(per_sensor["mae"], bins=50, color=BLUE, edgecolor=SURFACE, linewidth=0.5)
    a2.axvline(per_sensor["mae"].median(), color=INK, ls="--", lw=1,
               label=f"median {per_sensor['mae'].median():.1f}")
    a2.set_xlabel("Per-sensor test MAE"); a2.set_ylabel("Sensors")
    a2.set_title(f"Per-sensor MAE ({len(per_sensor)} sensors)"); a2.legend()
    fig.tight_layout(); fig.savefig(f"{out}/fig_test_errors.png", dpi=200); plt.show()

    # 4. Observed vs predicted, first 3 test days, median- and high-flow sensor
    times = pd.to_datetime(pred["timestamps_ns"])
    window = times < times[0] + pd.Timedelta(days=3)
    order = np.argsort(pred["targets"].mean(axis=0))
    picks = [order[len(order) // 2], order[int(0.9 * (len(order) - 1))]]
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    for a, j in zip(axes, picks):
        a.plot(times[window], pred["targets"][window, j], color=INK, lw=1.1, label="Observed")
        a.plot(times[window], pred["predictions"][window, j], color=BLUE, lw=1.0, label=label)
        a.set_title(f"Sensor {int(pred['sensor_ids'][j])}"); a.set_ylabel("Flow"); a.legend(loc="upper right")
    fig.tight_layout(); fig.savefig(f"{out}/fig_example_series.png", dpi=200); plt.show()

print("Figures saved in", out, "->", [f for f in os.listdir(out) if f.startswith("fig_")])


## 6. Clean up

Deletes the cloned `implementation` folder so the version's Output tab holds only `output_<RUN_NAME>`.

In [ ]:
import shutil, os

# The run folder was copied to output_<RUN_NAME> in section 4; the clone
# (code, dataset link, graphs, results) is not needed in the output.
assert os.path.exists(f"{out}/results.json"), "output folder missing - not deleting the clone"
shutil.rmtree(CODE)
print("Output contents:", sorted(os.listdir(WORK)))